# 向量存储（Vector Stores）
将文本向量化之后，下一步就是进行向量的存储

## 向量数据库
在向量数据库中进行检索时，检索并 不是唯一的、精确的 ，而是查询和目标向量 最为相似的一些向量 ，具有模糊性。

## 常用的向量数据库
FAISS

Chroma:开源、免费的 轻量级 向量数据库，有极简的 API

Milvus:开源的专为向量搜索设计的 云原生数据库 。性能强悍，功能丰富。覆盖 轻量级 的原型开发到 十亿级向量 的大规模生产系统

Redis

Elasticsearch

## Milvus数据模型
### 整体结构
Database -> Collection -> Partition -> Entity

- Database：Milvus 的数据库，用来隔离不同业务数据。

- Collection：最核心的逻辑容器，类似于关系型数据库里的 table。

- Partition：分区，是 collection 的子集，不是必须手动创建；一个 collection 至少会有默认partition。

- Entity：可以理解为 collection 中的一条记录。类似于关系型数据库的一行数据。


### 基本用法
#### DD操作
##### 查看数据库
1. 创建客户端
   ```python
   from pymilvus import MilvusClient
   client = MilvusClient("http://localhost:19530")
   ```

2. 列出所有数据库
   ```python
    existing_dbs = client.list_databases()
    print("== Databases ==")
    for db in existing_dbs:
        print(db)
   ```

3. 创建数据库
   ```python
    db_name = "rag_tutorial"
    if db_name not in existing_dbs:
        client.create_database(db_name=db_name)
   ```

4. 删除数据库
   ```python
   client.drop_database(db_name=db_name)
   ```

##### Collection

1. 切换数据库
   ```python
   client.use_database(db_name=db_name)
   ```

2. 查看数据库下的collections
   ```python
   collections = client.list_collections()
   print("== Collections ==")
   for collection in collections:
       print(collection)
   ```

3. 创建集合
   ```python
   collection_name = "rag_tutorial"
   client.create_collection(
        collection_name=collection_name,
        dimension=1024,
        metric_type="COSINE"
    )
   ```
- dimension ：嵌入向量维度，应和嵌入模型的向量维度保持一致，见下文。
  
- metric_type ：表示向量相似度的计算方式， COSINE 表示余弦相似度
- 
1. 删除集合
   ```python
   client.drop_collection(collection_name=collection_name)
   ```

### DML操作
#### 嵌入模型的初始化


In [2]:
from langchain.embeddings import init_embeddings
import os
from dotenv import load_dotenv
load_dotenv(override=True)

# 初始化嵌入模型
embed_model = init_embeddings(
    model="openai:Pro/BAAI/bge-m3",
    api_key=os.getenv("SILICONFLOW_API_KEY"),
    base_url=os.getenv("SILICONFLOW_BASE_URL"),
)

In [3]:
# 创建collection
from pymilvus import MilvusClient

client = MilvusClient("http://localhost:19530")

db_name = "rag_tutorial"
collection_name = "docs"

existing_dbs = client.list_databases() # 查看所有数据库

if db_name not in existing_dbs:
    client.create_database(db_name=db_name) # 创建数据库

client.use_database(db_name=db_name) # 切换到rag_tutorial数据库

client.create_collection(
    collection_name=collection_name,
    dimension=1024,
    metric_type="COSINE"
)


In [4]:
from rich import print as rprint
# 查看 collection 元数据
metadata = client.describe_collection(
    collection_name=collection_name,
)
rprint(metadata)

{
    'collection_name': 'docs',
    'auto_id': False,
    'num_shards': 1,
    'description': '',
    'fields': [
        {
            'field_id': 100,
            'name': 'id',
            'description': '',
            'type': <DataType.INT64: 5>,
            'params': {},
            'is_primary': True
        },
        {
            'field_id': 101,
            'name': 'vector',
            'description': '',
            'type': <DataType.FLOAT_VECTOR: 101>,
            'params': {'dim': 1024}
        }
    ],
    'functions': [],
    'aliases': [],
    'collection_id': 467621515001874210,
    'consistency_level': 2,
    'properties': {'timezone': 'UTC'},
    'num_partitions': 1,
    'enable_dynamic_field': True,
    'enable_namespace': False,
    'created_timestamp': 467622397229924371,
    'update_timestamp': 467622397229924371
}

我们在创建collection时没有指定schema，后者可以理解为表结构，此时Milvus会将collection定义为默认结构。字段信息如下:

- id ：数据ID，作为主键唯一标识数据
- vector ：数据的嵌入向量

此外， enable_dynamic_field 为 True ，这表示 collection 支持动态字段。也就是说，除了预定义的id 和 vector 字段之外，在插入数据时还可以携带其他未提前声明的字段，这些字段会被 自动写入 并 统一存储在动态字段 中。这样做的好处是能够在不修改 schema 的情况下，灵活保存额外的业务属性，例如文本内容、标签、时间戳或来源信息等，适合字段结构不固定的场景。

#### 准备数据
##### 准备原始数据

In [5]:
# 准备测试数据
texts = [
"LangChain 是一个用于构建 LLM 应用的开发框架。",
"Milvus 是一个适合 AI 应用的向量数据库。",
"RAG 的核心是先检索相关知识，再让大模型生成答案。",
"Docker Desktop 可以方便地在本地运行 Milvus Standalone。"
]

# 生成嵌入向量
vectors = embed_model.embed_documents(texts)

print(len(vectors))
print(len(vectors[0]))
print(vectors[0][:5])

# 封装为可以插入的数据格式
data = [
    {"id": i, "vector": vectors[i], "text": texts[i], "source": "demo"}
    for i in range(len(texts))
]

4
1024
[-0.00875447504222393, 0.049498990178108215, 0.04265338554978371, -0.004870911128818989, 0.007635482121258974]


In [6]:
#### 写入数据
##### 插入数据
# 插入数据
insert_res = client.upsert(
    collection_name=collection_name,
    data=data
) # upsert可以保证幂等写入，即主键相同时覆盖
print("insert result:", insert_res)

insert result: {'upsert_count': 4, 'ids': [0, 1, 2, 3]}


In [7]:
# 手动flush，Milvus不会第一时间将数据落盘，要看到写入效果，我们手动flush，将数据刷写到磁盘
client.flush(collection_name=collection_name)

In [8]:
# 查看 collection 统计
stats = client.get_collection_stats(collection_name=collection_name)
print("collection stats:", stats)

collection stats: {'row_count': 4}


### DQL操作
#### 扫描数据
通过 query_iterator 扫描collection下的所有数据

In [9]:
it = client.query_iterator(
    collection_name=collection_name,
    batch_size=100,
    filter="",  # 不加过滤 = 扫全部
    output_fields=["*"],  # 你的显式 schema 字段
)

i = 0
while True:
    rows = it.next()
    if not rows:
        break
    for row in rows:
        print("=" * 30, f"-> 第{i + 1}条 <-", "=" * 30)
        print(row)
        print("=" * 30, f"-> 第{i + 1}条 <-", "=" * 30)
        i += 1

it.close()

============================== -> 第1条 <- ==============================
{'id': 0, 'vector': [-0.00875447504222393, 0.049498990178108215, 0.04265338554978371, -0.004870911128818989, 0.007635482121258974, 0.030805222690105438, 0.035544488579034805, 0.050552159547805786, -0.0037683737464249134, -0.007635482121258974, -0.01092663872987032, 0.011387400329113007, -0.031595099717378616, -0.012243101373314857, -0.0008186750928871334, -0.015007671900093555, 0.011321577243506908, -0.041336920112371445, 0.007832951843738556, -0.059240810573101044, -0.01415197178721428, 0.0679294615983963, -0.018825413659214973, -0.004278502892702818, -0.011913985013961792, 0.05608130246400833, -0.005298761650919914, -0.017245657742023468, -0.03146345168352127, -0.02290644682943821, 0.028830528259277344, 0.022774800658226013, 0.028040651232004166, -0.03633436560630798, -0.003817741060629487, -0.013427916914224625, 0.01125575415790081, -0.007306366693228483, -0.06266361474990845, 0.02106340043246746, 0.003686094889

In [10]:
# 通过主键查询数据
# 查询 
res = client.get(
    collection_name=collection_name,
    ids=[0, 1, 2, 3],
)
print(len(res))
for i in range(len(res)):
    print("=" * 30, f'-> 第{i + 1}条 <-', "=" * 30)
    print(res[i])
    print("=" * 30, f'-> 第{i + 1}条 <-', "=" * 30)

4
============================== -> 第1条 <- ==============================
{'id': 0, 'vector': [-0.00875447504222393, 0.049498990178108215, 0.04265338554978371, -0.004870911128818989, 0.007635482121258974, 0.030805222690105438, 0.035544488579034805, 0.050552159547805786, -0.0037683737464249134, -0.007635482121258974, -0.01092663872987032, 0.011387400329113007, -0.031595099717378616, -0.012243101373314857, -0.0008186750928871334, -0.015007671900093555, 0.011321577243506908, -0.041336920112371445, 0.007832951843738556, -0.059240810573101044, -0.01415197178721428, 0.0679294615983963, -0.018825413659214973, -0.004278502892702818, -0.011913985013961792, 0.05608130246400833, -0.005298761650919914, -0.017245657742023468, -0.03146345168352127, -0.02290644682943821, 0.028830528259277344, 0.022774800658226013, 0.028040651232004166, -0.03633436560630798, -0.003817741060629487, -0.013427916914224625, 0.01125575415790081, -0.007306366693228483, -0.06266361474990845, 0.02106340043246746, 0.0036860948

In [11]:
# 相似度检索
## 准备查询嵌入
query = "什么是向量数据库？"
query_vector = embed_model.embed_query(query)

## 检索
results = client.search(
    collection_name=collection_name,
    data=[query_vector], # Milvus search 这里仍然要传二维列表
    limit=3,
    output_fields=["text", "source"]
)
print("=== search results ===")
for hit in results[0]:
    print(hit)

=== search results ===
{'id': 0, 'distance': 0.6406217813491821, 'entity': {'text': 'LangChain 是一个用于构建 LLM 应用的开发框架。', 'source': 'demo'}}
{'id': 3, 'distance': 0.3116165101528168, 'entity': {'text': 'Docker Desktop 可以方便地在本地运行 Milvus Standalone。', 'source': 'demo'}}
{'id': 2, 'distance': 0.30259355902671814, 'entity': {'text': 'RAG 的核心是先检索相关知识，再让大模型生成答案。', 'source': 'demo'}}


- limit 表示检索结果最多保留几条数据
- output_fields 表示输出的实体中展示哪些字段